In [1]:
import os
import ROOT
import pandas as pd
import numpy as np

# =====================================================
# WORKING DIRECTORY AND SETTINGS
# =====================================================
working_directory = "/root/geant4/detector/Lung_ICRP"
os.chdir(working_directory)

file_path = "lung_TissueTumor_6.root"
tree_name = "t"

selected_volumes = [2, 3, 400]
selected_pdg = 22

excel_file = "lung_TissueTumor_400_all_gamma_data.xlsx"

print("Working directory:", os.getcwd())
print("Input file:", os.path.abspath(file_path))

# =====================================================
# OPEN ROOT FILE
# =====================================================
root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise OSError(f"Could not open {os.path.abspath(file_path)}")

tree = root_file.Get(tree_name)

if not tree:
    root_file.Close()
    raise KeyError(f"TTree '{tree_name}' not found")

print("Total TTree entries:", tree.GetEntries())

# =====================================================
# SAFE VECTOR READER
# =====================================================
def safe_value(vector, index, default=np.nan):
    """
    Return vector[index] when available.
    Otherwise return NaN.
    """
    if index < len(vector):
        return vector[index]
    return default

# =====================================================
# STORAGE
# =====================================================
volume_data = {
    volume: [] for volume in selected_volumes
}

combined_data = []

column_order = [
    "tree_entry",
    "vector_index",
    "vlm",
    "pdg",
    "pro",
    "stp",
    "trk",
    "k",
    "et",
    "de",
    "x",
    "y",
    "z",
    "px",
    "py",
    "pz"
]

# =====================================================
# LOOP THROUGH ROOT DATA
# =====================================================
for tree_entry, event in enumerate(tree):

    # Use pdg as the principal record length
    number_of_records = len(event.pdg)

    for vector_index in range(number_of_records):

        pdg_value = int(event.pdg[vector_index])

        # vlm must exist for this record
        if vector_index >= len(event.vlm):
            continue

        volume_value = int(event.vlm[vector_index])

        # Keep only gammas
        if pdg_value != selected_pdg:
            continue

        # Keep only detector volumes
        if volume_value not in selected_volumes:
            continue

        row = {
            "tree_entry": tree_entry,
            "vector_index": vector_index,

            "vlm": volume_value,
            "pdg": pdg_value,

            "pro": int(safe_value(event.pro, vector_index, -1)),
            "stp": int(safe_value(event.stp, vector_index, -1)),
            "trk": int(safe_value(event.trk, vector_index, -1)),

            "k": float(safe_value(event.k, vector_index)),
            "et": float(safe_value(event.et, vector_index)),
            "de": float(safe_value(event.de, vector_index)),

            "x": float(safe_value(event.x, vector_index)),
            "y": float(safe_value(event.y, vector_index)),
            "z": float(safe_value(event.z, vector_index)),

            "px": float(safe_value(event.px, vector_index)),
            "py": float(safe_value(event.py, vector_index)),
            "pz": float(safe_value(event.pz, vector_index))
        }

        volume_data[volume_value].append(row)
        combined_data.append(row)

# =====================================================
# CREATE DATAFRAMES
# =====================================================
dataframes = {
    volume: pd.DataFrame(
        volume_data[volume],
        columns=column_order
    )
    for volume in selected_volumes
}

df_all = pd.DataFrame(
    combined_data,
    columns=column_order
)

# =====================================================
# WRITE EXCEL WORKBOOK
# =====================================================
with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:

    for volume in selected_volumes:
        dataframes[volume].to_excel(
            writer,
            sheet_name=f"Volume_{volume}",
            index=False
        )

    df_all.to_excel(
        writer,
        sheet_name="All_Volumes",
        index=False
    )

# =====================================================
# CONFIRM OUTPUT
# =====================================================
print("\nRecords saved:")

for volume in selected_volumes:
    print(f"Volume {volume}: {len(dataframes[volume])}")

print("All volumes:", len(df_all))

print("\nExcel file saved at:")
print(os.path.abspath(excel_file))

print("File exists:", os.path.exists(excel_file))

if os.path.exists(excel_file):
    print("File size:", os.path.getsize(excel_file), "bytes")

display(df_all.head(10))

root_file.Close()


Working directory: /root/geant4/detector/Lung_ICRP
Input file: /root/geant4/detector/Lung_ICRP/lung_TissueTumor_6.root
Total TTree entries: 100000

Records saved:
Volume 2: 203166
Volume 3: 103218
Volume 400: 133944
All volumes: 440328

Excel file saved at:
/root/geant4/detector/Lung_ICRP/lung_TissueTumor_400_all_gamma_data.xlsx
File exists: True
File size: 55691190 bytes


,tree_entry,vector_index,vlm,pdg,pro,stp,trk,k,et,de,x,y,z,px,py,pz
0,0,2,2,22,1092,2,1,662.000000,NaN,0.0,0.0,2.500000,0.0,0.000000,-1.000000,0.000000
1,0,3,3,22,1092,3,1,662.000000,NaN,0.0,0.0,-2.500000,0.0,0.000000,-1.000000,0.000000
2,0,4,2,22,1092,4,1,662.000000,NaN,0.0,0.0,-10.000000,0.0,0.000000,-1.000000,0.000000
3,0,6,400,22,1092,6,1,662.000000,NaN,0.0,0.0,-55.000000,0.0,0.000000,-1.000000,0.000000
4,1,2,2,22,1092,2,1,662.000000,NaN,0.0,0.0,2.500000,0.0,0.000000,-1.000000,0.000000
5,1,3,3,22,1092,3,1,662.000000,NaN,0.0,0.0,-2.500000,0.0,0.000000,-1.000000,0.000000
6,1,4,2,22,1092,4,1,662.000000,NaN,0.0,0.0,-10.000000,0.0,0.000000,-1.000000,0.000000
7,1,6,400,22,1092,6,1,662.000000,NaN,0.0,0.0,-55.000000,0.0,0.000000,-1.000000,0.000000
8,2,2,2,22,1092,2,1,662.000000,0.000000,0.0,0.0,2.500000,0.0,0.000000,-1.000000,0.000000
9,2,3,3,22,2013,3,1,269.253878,392.746122,0.0,0.0,0.645375,0.0,0.732468,0.125931,0.669053


In [1]:
import os
import ROOT
import pandas as pd
import numpy as np

# =====================================================
# USER SETTINGS
# =====================================================
working_directory = "/root/geant4/detector/Lung_ICRP"
os.chdir(working_directory)

file_path = "lung_TissueTumor_6.root"
tree_name = "t"

# Tissue/tumor volumes
scatter_volumes = [2, 3]

# Detector volume
detector_volume = 400

# Particle/process
gamma_pdg = 22
compton_process = 2013

excel_file = "gamma_vlm2_3_to_detector400.xlsx"

print("Working directory:", os.getcwd())
print("Input file:", os.path.abspath(file_path))


# =====================================================
# OPEN ROOT FILE
# =====================================================
root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise OSError(f"Could not open {os.path.abspath(file_path)}")

tree = root_file.Get(tree_name)

if not tree:
    root_file.Close()
    raise KeyError(f"TTree '{tree_name}' not found")

print("Total TTree entries:", tree.GetEntries())


# =====================================================
# SAFE VECTOR READER
# =====================================================
def safe_value(vector, index, default=np.nan):
    if index < len(vector):
        return vector[index]
    return default


# =====================================================
# STORAGE
# =====================================================
matched_rows = []


# =====================================================
# LOOP OVER TTREE ENTRIES
# =====================================================
for tree_entry, event in enumerate(tree):

    n_records = len(event.pdg)

    # -------------------------------------------------
    # STEP 1:
    # Find Compton-scattering gamma records in
    # tissue/tumor (vlm 2 or 3)
    # -------------------------------------------------
    scatter_candidates = []

    for i in range(n_records):

        # Make sure required vectors contain index i
        if i >= len(event.vlm):
            continue

        pdg_i = int(event.pdg[i])
        vlm_i = int(event.vlm[i])

        # Gamma only
        if pdg_i != gamma_pdg:
            continue

        # Tissue/tumor only
        if vlm_i not in scatter_volumes:
            continue

        pro_i = int(safe_value(event.pro, i, -1))

        # Compton scattering only
        if pro_i != compton_process:
            continue

        trk_i = int(safe_value(event.trk, i, -1))

        # Invalid/missing track ID
        if trk_i < 0:
            continue

        scatter_candidates.append({
            "index": i,
            "trk": trk_i,
            "vlm": vlm_i,
            "pro": pro_i,

            "stp": int(safe_value(event.stp, i, -1)),

            "k": float(safe_value(event.k, i)),
            "et": float(safe_value(event.et, i)),
            "de": float(safe_value(event.de, i)),

            "x": float(safe_value(event.x, i)),
            "y": float(safe_value(event.y, i)),
            "z": float(safe_value(event.z, i)),

            "px": float(safe_value(event.px, i)),
            "py": float(safe_value(event.py, i)),
            "pz": float(safe_value(event.pz, i))
        })


    # -------------------------------------------------
    # STEP 2:
    # For each scattering candidate, search later
    # records for SAME gamma track in detector 400
    # -------------------------------------------------
    for scatter in scatter_candidates:

        scatter_index = scatter["index"]
        scatter_trk = scatter["trk"]

        detector_match = None

        # Start AFTER the scattering record
        for j in range(scatter_index + 1, n_records):

            if j >= len(event.vlm):
                continue

            pdg_j = int(event.pdg[j])
            vlm_j = int(event.vlm[j])

            # Gamma only
            if pdg_j != gamma_pdg:
                continue

            # Must be detector 400
            if vlm_j != detector_volume:
                continue

            trk_j = int(safe_value(event.trk, j, -1))

            # Must be SAME gamma track
            if trk_j != scatter_trk:
                continue

            # We found the same gamma in detector 400
            detector_match = {
                "index": j,
                "trk": trk_j,

                "pro": int(safe_value(event.pro, j, -1)),
                "stp": int(safe_value(event.stp, j, -1)),

                "k": float(safe_value(event.k, j)),
                "et": float(safe_value(event.et, j)),
                "de": float(safe_value(event.de, j)),

                "x": float(safe_value(event.x, j)),
                "y": float(safe_value(event.y, j)),
                "z": float(safe_value(event.z, j)),

                "px": float(safe_value(event.px, j)),
                "py": float(safe_value(event.py, j)),
                "pz": float(safe_value(event.pz, j))
            }

            # Take first detector-400 record
            break


        # -------------------------------------------------
        # STEP 3:
        # Save only if detector 400 was reached
        # -------------------------------------------------
        if detector_match is None:
            continue


        # -------------------------------------------------
        # Calculate scattering angle from momentum
        #
        # Incident gamma direction = (0,-1,0)
        #
        # cos(theta) = -py / |p|
        # -------------------------------------------------
        px = scatter["px"]
        py = scatter["py"]
        pz = scatter["pz"]

        p_mag = np.sqrt(px**2 + py**2 + pz**2)

        if np.isfinite(p_mag) and p_mag > 0:

            cos_theta = -py / p_mag
            cos_theta = np.clip(cos_theta, -1.0, 1.0)

            theta_p_deg = np.degrees(np.arccos(cos_theta))

        else:
            theta_p_deg = np.nan


        # -------------------------------------------------
        # Save scatter + detector information in SAME ROW
        # -------------------------------------------------
        row = {

            # Event/track identification
            "tree_entry": tree_entry,
            "trk": scatter_trk,

            # =============================================
            # SCATTER INFORMATION
            # =============================================
            "scatter_index": scatter["index"],
            "scatter_vlm": scatter["vlm"],
            "scatter_pro": scatter["pro"],
            "scatter_stp": scatter["stp"],

            "scatter_k": scatter["k"],
            "scatter_et": scatter["et"],
            "scatter_de": scatter["de"],

            "scatter_x": scatter["x"],
            "scatter_y": scatter["y"],
            "scatter_z": scatter["z"],

            "scatter_px": scatter["px"],
            "scatter_py": scatter["py"],
            "scatter_pz": scatter["pz"],

            "theta_p_deg": theta_p_deg,

            # =============================================
            # DETECTOR INFORMATION
            # =============================================
            "detector_vlm": detector_volume,
            "detector_index": detector_match["index"],
            "detector_pro": detector_match["pro"],
            "detector_stp": detector_match["stp"],

            "detector_k": detector_match["k"],
            "detector_et": detector_match["et"],
            "detector_de": detector_match["de"],

            "detector_x": detector_match["x"],
            "detector_y": detector_match["y"],
            "detector_z": detector_match["z"],

            "detector_px": detector_match["px"],
            "detector_py": detector_match["py"],
            "detector_pz": detector_match["pz"]
        }

        matched_rows.append(row)


# =====================================================
# CREATE DATAFRAME
# =====================================================
df = pd.DataFrame(matched_rows)


# =====================================================
# SPLIT VOLUME 2 AND VOLUME 3
# =====================================================
if len(df) > 0:

    df_vlm2 = df[df["scatter_vlm"] == 2].copy()
    df_vlm3 = df[df["scatter_vlm"] == 3].copy()

else:

    df_vlm2 = pd.DataFrame()
    df_vlm3 = pd.DataFrame()


# =====================================================
# WRITE EXCEL FILE
# =====================================================
with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:

    df.to_excel(
        writer,
        sheet_name="All_Matched",
        index=False
    )

    df_vlm2.to_excel(
        writer,
        sheet_name="Scatter_Volume_2",
        index=False
    )

    df_vlm3.to_excel(
        writer,
        sheet_name="Scatter_Volume_3",
        index=False
    )


# =====================================================
# SUMMARY
# =====================================================
print("\n========================================")
print("MATCHING SUMMARY")
print("========================================")

print("Total matched gammas:", len(df))
print("Scattered in volume 2:", len(df_vlm2))
print("Scattered in volume 3:", len(df_vlm3))

print("\nSelection:")
print("PDG              =", gamma_pdg)
print("Compton process  =", compton_process)
print("Scatter volumes  =", scatter_volumes)
print("Detector volume  =", detector_volume)

print("\nExcel file:")
print(os.path.abspath(excel_file))

print("File exists:", os.path.exists(excel_file))

if os.path.exists(excel_file):
    print("File size:", os.path.getsize(excel_file), "bytes")


# =====================================================
# DISPLAY FIRST RECORDS
# =====================================================
display(df.head(20))


# =====================================================
# CLOSE ROOT FILE
# =====================================================
root_file.Close()

Working directory: /root/geant4/detector/Lung_ICRP
Input file: /root/geant4/detector/Lung_ICRP/lung_TissueTumor_6.root
Total TTree entries: 100000


ModuleNotFoundError: No module named 'openpyxl'

In [1]:
import os
import ROOT
import pandas as pd
import numpy as np

# =====================================================
# SETTINGS
# =====================================================
working_directory = "/root/geant4/detector/Lung_ICRP"
os.chdir(working_directory)

file_path = "lung_TissueTumor_430.root"
tree_name = "t"

# Incident gamma energy, keV
E0 = 662.0

# Electron rest energy, keV
ME = 511.0

# Incident beam direction
# /gps/direction 0 -1 0
incident_dir = np.array([0.0, -1.0, 0.0])

# Volumes where scattering of interest occurs
scatter_volumes = [2, 3]

# Detector volume
detector_volume = 430

# Gamma PDG
gamma_pdg = 22

# Your Compton process code
compton_process = 2013

# Output
excel_file = "gamma_vlm2_3_to_detector430_1.xlsx"

print("Working directory:", os.getcwd())
print("ROOT file:", os.path.abspath(file_path))


# =====================================================
# OPEN ROOT FILE
# =====================================================
root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise OSError(
        f"Could not open ROOT file: {os.path.abspath(file_path)}"
    )

tree = root_file.Get(tree_name)

if not tree:
    root_file.Close()
    raise KeyError(
        f"TTree '{tree_name}' not found"
    )

print("Total TTree entries:", tree.GetEntries())


# =====================================================
# SAFE VECTOR VALUE
# =====================================================
def safe_value(vector, index, default=np.nan):
    if index < len(vector):
        return vector[index]
    return default


# =====================================================
# ANGLE FROM MOMENTUM DIRECTION
# =====================================================
def theta_from_momentum(px, py, pz):

    pvec = np.array(
        [px, py, pz],
        dtype=float
    )

    pmag = np.linalg.norm(pvec)

    if pmag == 0:
        return np.nan

    cos_theta = np.dot(
        pvec,
        incident_dir
    ) / pmag

    cos_theta = np.clip(
        cos_theta,
        -1.0,
        1.0
    )

    return np.degrees(
        np.arccos(cos_theta)
    )


# =====================================================
# ANGLE FROM COMPTON ENERGY
# =====================================================
def theta_from_k(k):

    if not np.isfinite(k):
        return np.nan

    if k <= 0:
        return np.nan

    cos_theta = (
        1.0
        - ME
        * (
            (1.0 / k)
            - (1.0 / E0)
        )
    )

    # If outside physical single-Compton range
    if cos_theta < -1.0 or cos_theta > 1.0:
        return np.nan

    return np.degrees(
        np.arccos(cos_theta)
    )


# =====================================================
# STORAGE
# =====================================================
matched_rows = []


# =====================================================
# LOOP THROUGH EVENTS
# =====================================================
for tree_entry, event in enumerate(tree):

    n = len(event.pdg)

    # -------------------------------------------------
    # Collect all records for this TTree entry
    # -------------------------------------------------
    records = []

    for i in range(n):

        if i >= len(event.vlm):
            continue

        if i >= len(event.trk):
            continue

        pdg = int(event.pdg[i])
        vlm = int(event.vlm[i])
        trk = int(event.trk[i])

        record = {
            "index": i,
            "pdg": pdg,
            "vlm": vlm,
            "trk": trk,

            "pro": int(
                safe_value(
                    event.pro,
                    i,
                    -1
                )
            ),

            "stp": int(
                safe_value(
                    event.stp,
                    i,
                    -1
                )
            ),

            "k": float(
                safe_value(
                    event.k,
                    i
                )
            ),

            "et": float(
                safe_value(
                    event.et,
                    i
                )
            ),

            "de": float(
                safe_value(
                    event.de,
                    i
                )
            ),

            "x": float(
                safe_value(
                    event.x,
                    i
                )
            ),

            "y": float(
                safe_value(
                    event.y,
                    i
                )
            ),

            "z": float(
                safe_value(
                    event.z,
                    i
                )
            ),

            "px": float(
                safe_value(
                    event.px,
                    i
                )
            ),

            "py": float(
                safe_value(
                    event.py,
                    i
                )
            ),

            "pz": float(
                safe_value(
                    event.pz,
                    i
                )
            )
        }

        records.append(record)

    # =================================================
    # FIND COMPTON SCATTER RECORDS IN VOLUME 2 OR 3
    # =================================================
    scatter_records = []

    for rec in records:

        if rec["pdg"] != gamma_pdg:
            continue

        if rec["vlm"] not in scatter_volumes:
            continue

        if rec["pro"] != compton_process:
            continue

        scatter_records.append(rec)

    # =================================================
    # FOR EACH SCATTER, FIND SAME TRACK LATER IN VLM 430
    # =================================================
    for scatter in scatter_records:

        scatter_index = scatter["index"]
        scatter_track = scatter["trk"]

        detector_candidates = []

        for rec in records:

            # Must be later in the stored sequence
            if rec["index"] <= scatter_index:
                continue

            # Same gamma track
            if rec["trk"] != scatter_track:
                continue

            if rec["pdg"] != gamma_pdg:
                continue

            # Must reach detector 430
            if rec["vlm"] != detector_volume:
                continue

            detector_candidates.append(rec)

        # No detector hit from this scattered gamma
        if len(detector_candidates) == 0:
            continue

        # -------------------------------------------------
        # Use first occurrence in detector
        # -------------------------------------------------
        det = detector_candidates[0]

        # =================================================
        # CALCULATE ANGLES AT SCATTER
        # =================================================
        theta_p_scatter = theta_from_momentum(
            scatter["px"],
            scatter["py"],
            scatter["pz"]
        )

        theta_k_scatter = theta_from_k(
            scatter["k"]
        )

        # =================================================
        # CALCULATE ANGLE FROM MOMENTUM AT DETECTOR
        # =================================================
        theta_p_detector = theta_from_momentum(
            det["px"],
            det["py"],
            det["pz"]
        )

        theta_k_detector = theta_from_k(
            det["k"]
        )

        # =================================================
        # ENERGY LOSS
        # =================================================
        energy_loss_scatter = (
            E0 - scatter["k"]
            if np.isfinite(scatter["k"])
            else np.nan
        )

        # =================================================
        # SAVE MATCH
        # =================================================
        matched_rows.append({

            # Event identity
            "tree_entry": tree_entry,
            "trk": scatter_track,

            # ---------------------------------------------
            # SCATTER RECORD
            # ---------------------------------------------
            "scatter_volume": scatter["vlm"],
            "scatter_vector_index": scatter["index"],
            "scatter_pro": scatter["pro"],
            "scatter_stp": scatter["stp"],

            "scatter_k_keV": scatter["k"],
            "scatter_energy_loss_keV":
                energy_loss_scatter,

            "scatter_et": scatter["et"],
            "scatter_de": scatter["de"],

            "scatter_x": scatter["x"],
            "scatter_y": scatter["y"],
            "scatter_z": scatter["z"],

            "scatter_px": scatter["px"],
            "scatter_py": scatter["py"],
            "scatter_pz": scatter["pz"],

            "theta_p_scatter_deg":
                theta_p_scatter,

            "theta_k_scatter_deg":
                theta_k_scatter,

            # ---------------------------------------------
            # DETECTOR RECORD
            # ---------------------------------------------
            "detector_volume": detector_volume,
            "detector_vector_index": det["index"],
            "detector_pro": det["pro"],
            "detector_stp": det["stp"],

            "detector_k_keV": det["k"],
            "detector_et": det["et"],
            "detector_de": det["de"],

            "detector_x": det["x"],
            "detector_y": det["y"],
            "detector_z": det["z"],

            "detector_px": det["px"],
            "detector_py": det["py"],
            "detector_pz": det["pz"],

            "theta_p_detector_deg":
                theta_p_detector,

            "theta_k_detector_deg":
                theta_k_detector,

            # ---------------------------------------------
            # ANGLE COMPARISON
            # ---------------------------------------------
            "theta_difference_scatter_deg":
                (
                    theta_p_scatter
                    - theta_k_scatter
                    if np.isfinite(theta_p_scatter)
                    and np.isfinite(theta_k_scatter)
                    else np.nan
                ),

            "theta_difference_detector_deg":
                (
                    theta_p_detector
                    - theta_k_detector
                    if np.isfinite(theta_p_detector)
                    and np.isfinite(theta_k_detector)
                    else np.nan
                )
        })


# =====================================================
# CREATE DATAFRAME
# =====================================================
df_matches = pd.DataFrame(matched_rows)

print("\n========================================")
print("MATCHED GAMMAS")
print("========================================")

print(
    "Total gammas scattered in volume 2 or 3 "
    "and later reaching volume 430:",
    len(df_matches)
)


# =====================================================
# SEPARATE VOLUME 2 AND 3
# =====================================================
if len(df_matches) > 0:

    df_vlm2 = df_matches[
        df_matches["scatter_volume"] == 2
    ].copy()

    df_vlm3 = df_matches[
        df_matches["scatter_volume"] == 3
    ].copy()

else:

    df_vlm2 = pd.DataFrame()
    df_vlm3 = pd.DataFrame()


print("From volume 2:", len(df_vlm2))
print("From volume 3:", len(df_vlm3))


# =====================================================
# OPTIONAL: DETECTOR ANGULAR ACCEPTANCE
# 30 +/- 11 degrees
# =====================================================
theta_min = 19.0
theta_max = 41.0

if len(df_matches) > 0:

    df_detector_angle = df_matches[
        (df_matches["theta_p_detector_deg"] >= theta_min)
        &
        (df_matches["theta_p_detector_deg"] <= theta_max)
    ].copy()

else:

    df_detector_angle = pd.DataFrame()


print(
    "Matched photons with detector direction "
    "between 19 and 41 deg:",
    len(df_detector_angle)
)


# =====================================================
# OPTIONAL: CLEAN SINGLE-SCATTER-LIKE SAMPLE
#
# Require theta from momentum and theta from k
# to approximately agree.
# =====================================================
angle_tolerance = 5.0   # degrees

if len(df_matches) > 0:

    df_single_like = df_matches[
        np.isfinite(
            df_matches["theta_k_scatter_deg"]
        )
        &
        (
            np.abs(
                df_matches["theta_p_scatter_deg"]
                -
                df_matches["theta_k_scatter_deg"]
            )
            <= angle_tolerance
        )
    ].copy()

else:

    df_single_like = pd.DataFrame()


print(
    "Single-Compton-like matches "
    f"(|theta_p-theta_k| <= {angle_tolerance} deg):",
    len(df_single_like)
)


# =====================================================
# SAVE EXCEL
# =====================================================
with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    df_matches.to_excel(
        writer,
        sheet_name="All_matches",
        index=False
    )

    df_vlm2.to_excel(
        writer,
        sheet_name="Scatter_Volume_2",
        index=False
    )

    df_vlm3.to_excel(
        writer,
        sheet_name="Scatter_Volume_3",
        index=False
    )

    df_detector_angle.to_excel(
        writer,
        sheet_name="Detector_19_41deg",
        index=False
    )

    df_single_like.to_excel(
        writer,
        sheet_name="Single_Compton_like",
        index=False
    )


# =====================================================
# DISPLAY
# =====================================================
print("\nExcel file saved:")
print(os.path.abspath(excel_file))

if len(df_matches) > 0:

    display(
        df_matches[
            [
                "tree_entry",
                "trk",
                "scatter_volume",
                "scatter_k_keV",
                "theta_p_scatter_deg",
                "theta_k_scatter_deg",
                "detector_k_keV",
                "theta_p_detector_deg",
                "theta_k_detector_deg"
            ]
        ].head(30)
    )

else:

    print(
        "\nNo matching gamma tracks were found."
    )


# =====================================================
# CLOSE ROOT FILE
# =====================================================
root_file.Close()

Working directory: /root/geant4/detector/Lung_ICRP
ROOT file: /root/geant4/detector/Lung_ICRP/lung_TissueTumor_430.root
Total TTree entries: 100000

MATCHED GAMMAS
Total gammas scattered in volume 2 or 3 and later reaching volume 430: 250
From volume 2: 120
From volume 3: 130
Matched photons with detector direction between 19 and 41 deg: 176
Single-Compton-like matches (|theta_p-theta_k| <= 5.0 deg): 240

Excel file saved:
/root/geant4/detector/Lung_ICRP/gamma_vlm2_3_to_detector430_1.xlsx


,tree_entry,trk,scatter_volume,scatter_k_keV,theta_p_scatter_deg,theta_k_scatter_deg,detector_k_keV,theta_p_detector_deg,theta_k_detector_deg
0,120,1,2,628.280052,16.549935,16.549953,628.280052,16.549935,16.549953
1,1778,1,2,532.641434,35.655734,35.655773,532.641434,35.655734,35.655773
2,1895,1,2,600.283223,22.980372,22.980397,600.283223,22.980372,22.980397
3,3584,1,3,302.460210,85.271925,85.272038,170.036151,40.113802,NaN
4,3584,1,3,170.036151,40.113802,NaN,170.036151,40.113802,NaN
5,4039,1,3,539.099319,34.510092,34.510130,442.344840,4.507621,51.924742
6,4227,1,2,538.846022,34.555127,34.555165,538.846022,34.555127,34.555165
7,5637,1,2,496.553569,42.028622,42.028669,496.553569,42.028622,42.028669
8,5880,1,3,565.506955,29.739515,29.739547,384.474531,84.285790,63.716271
9,5938,1,3,589.262326,25.214726,25.214753,589.262326,25.214726,25.214753
